# Sistema de Análisis y Gestión de Riesgo Bancario

Notebook único para un prototipo académico inspirado en un contexto BBVA.

Incluye:
- generación de datos sintéticos,
- entrenamiento y evaluación de un modelo de riesgo,
- registro opcional con MLflow,
- trazabilidad y logs estructurados,
- métricas operativas y de modelo,
- detección de data drift,
- SLO y error budgets,
- motor de alertas,
- simulación de incidentes,
- runbooks automáticos,
- dashboard interactivo dentro del notebook.


In [13]:
%pip -q install numpy pandas scikit-learn plotly ipywidgets pyyaml joblib
%pip -q install fastapi uvicorn pydantic prometheus-client
%pip -q install mlflow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1) Imports, configuración y almacenes en memoria

In [14]:
from __future__ import annotations

import json
import math
import os
import platform
import shutil
import tempfile
import time
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Literal

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, clear_output

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    import mlflow
    import mlflow.sklearn
    MLflow_AVAILABLE = True
except Exception:
    mlflow = None
    MLflow_AVAILABLE = False

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception:
    widgets = None
    WIDGETS_AVAILABLE = False

try:
    from fastapi import FastAPI, Request
    from fastapi.responses import JSONResponse
    from pydantic import BaseModel, Field
    FASTAPI_AVAILABLE = True
except Exception:
    FastAPI = None
    Request = None
    JSONResponse = None
    BaseModel = object
    Field = lambda *args, **kwargs: None
    FASTAPI_AVAILABLE = False

np.random.seed(42)

CONFIG = {
    "project_name": "Sistema de Analisis y Gestion de Riesgo Bancario",
    "organization_context": "BBVA - Simulacion academica",
    "model_threshold": 0.50,
    "availability_target": 0.995,
    "latency_p95_ms_max": 500,
    "error_rate_max": 0.01,
    "roc_auc_min": 0.75,
    "drift_score_max": 0.50,
    "minimum_drift_samples": 80,
    "minimum_performance_samples": 40,
    "error_budget_period_hours": 24,
    "auto_response_enabled": True,
}

NUMERIC_FEATURES = [
    "age",
    "monthly_income",
    "debt_ratio",
    "credit_utilization",
    "months_employed",
    "delinquencies_12m",
    "credit_history_months",
    "requested_amount",
    "term_months",
    "num_open_accounts",
]

CATEGORICAL_FEATURES = ["region", "channel", "customer_segment"]
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET_COLUMN = "defaulted"

# Almacenes en memoria
stores = {
    "predictions": [],
    "logs": [],
    "traces": [],
    "alerts": [],
    "incidents": [],
    "runbooks": [],
    "monitoring_snapshots": [],
    "model_registry": [],
}

# Directorio temporal local para artefactos opcionales
BASE_DIR = Path(tempfile.gettempdir()) / "bank_risk_observability_notebook"
BASE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MLFLOW_DIR = BASE_DIR / "mlruns"
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)

def utcnow() -> str:
    return datetime.now(timezone.utc).isoformat()

def new_id(prefix: str = "") -> str:
    return f"{prefix}{uuid.uuid4().hex[:12]}"

def sigmoid(x):
    x = np.clip(x, -30, 30)
    return 1.0 / (1.0 + np.exp(-x))

print("Entorno listo:", platform.python_version())
print("MLflow disponible:", MLflow_AVAILABLE)
print("ipywidgets disponible:", WIDGETS_AVAILABLE)
print("FastAPI disponible:", FASTAPI_AVAILABLE)

Entorno listo: 3.12.0
MLflow disponible: True
ipywidgets disponible: True
FastAPI disponible: True


## 2) Generación de datos sintéticos para riesgo de crédito

In [15]:
def latent_probability(frame: pd.DataFrame) -> np.ndarray:
    income = frame["monthly_income"].astype(float).clip(lower=1)
    amount_income_ratio = frame["requested_amount"].astype(float) / (income * 12)

    region_effect = frame["region"].map({
        "Norte": 0.05,
        "Centro": 0.0,
        "Occidente": 0.08,
        "Sureste": 0.16,
    }).fillna(0)

    channel_effect = frame["channel"].map({
        "Sucursal": 0.0,
        "Digital": 0.08,
        "Movil": 0.12,
    }).fillna(0)

    segment_effect = frame["customer_segment"].map({
        "Masivo": 0.18,
        "Premium": -0.28,
        "PyME": 0.12,
    }).fillna(0)

    logit = (
        -3.25
        + 2.45 * frame["debt_ratio"].astype(float)
        + 2.10 * frame["credit_utilization"].astype(float)
        + 0.52 * frame["delinquencies_12m"].astype(float)
        + 0.90 * amount_income_ratio.clip(upper=4.0)
        - 0.0065 * frame["months_employed"].astype(float)
        - 0.0035 * frame["credit_history_months"].astype(float)
        + 0.010 * (frame["term_months"].astype(float) - 36.0)
        + 0.025 * (frame["num_open_accounts"].astype(float) - 5.0)
        + 0.010 * np.maximum(25.0 - frame["age"].astype(float), 0.0)
        + region_effect
        + channel_effect
        + segment_effect
    )
    return np.asarray(sigmoid(logit), dtype=float)

def generate_synthetic_data(n_samples: int = 8000, seed: int = 42, mode: str = "normal") -> pd.DataFrame:
    if n_samples <= 0:
        raise ValueError("n_samples debe ser mayor que cero.")

    rng = np.random.default_rng(seed)
    income_multiplier = 1.0
    debt_shift = 0.0
    utilization_shift = 0.0
    delinquency_lambda = 0.28
    amount_multiplier = 1.0

    if mode == "drift":
        income_multiplier = 0.72
        debt_shift = 0.16
        utilization_shift = 0.18
        delinquency_lambda = 0.65
        amount_multiplier = 1.18
    elif mode == "severe_drift":
        income_multiplier = 0.55
        debt_shift = 0.30
        utilization_shift = 0.30
        delinquency_lambda = 1.10
        amount_multiplier = 1.45

    age = rng.integers(18, 76, size=n_samples)
    monthly_income = rng.lognormal(mean=np.log(28000), sigma=0.55, size=n_samples) * income_multiplier
    debt_ratio = np.clip(rng.beta(a=2.0, b=4.0, size=n_samples) + debt_shift, 0, 1.7)
    credit_utilization = np.clip(rng.beta(a=1.8, b=3.5, size=n_samples) + utilization_shift, 0, 1.4)
    months_employed = np.clip(rng.gamma(shape=2.5, scale=28.0, size=n_samples), 0, 420).astype(int)
    delinquencies = np.clip(rng.poisson(lam=delinquency_lambda, size=n_samples), 0, 12)
    credit_history = np.clip(age * 7 + rng.normal(loc=0, scale=48, size=n_samples), 0, 600).astype(int)
    requested_amount = np.clip(
        rng.lognormal(mean=np.log(220000), sigma=0.72, size=n_samples) * amount_multiplier,
        5000,
        4_000_000,
    )

    frame = pd.DataFrame({
        "age": age,
        "monthly_income": monthly_income.round(2),
        "debt_ratio": debt_ratio.round(4),
        "credit_utilization": credit_utilization.round(4),
        "months_employed": months_employed,
        "delinquencies_12m": delinquencies,
        "credit_history_months": credit_history,
        "requested_amount": requested_amount.round(2),
        "term_months": rng.choice([12, 18, 24, 36, 48, 60, 72], size=n_samples),
        "num_open_accounts": np.clip(rng.poisson(lam=5.0, size=n_samples), 0, 25),
        "region": rng.choice(["Norte", "Centro", "Occidente", "Sureste"], size=n_samples, p=[0.23, 0.38, 0.22, 0.17]),
        "channel": rng.choice(["Sucursal", "Digital", "Movil"], size=n_samples, p=[0.42, 0.34, 0.24]),
        "customer_segment": rng.choice(["Masivo", "Premium", "PyME"], size=n_samples, p=[0.66, 0.19, 0.15]),
    })

    p = latent_probability(frame)
    log_odds = np.log(np.clip(p, 1e-8, 1 - 1e-8) / (1 - np.clip(p, 1e-8, 1 - 1e-8)))
    noise = rng.normal(loc=0, scale=0.25, size=n_samples)
    probability = np.asarray(sigmoid(log_odds + noise), dtype=float)
    frame[TARGET_COLUMN] = rng.binomial(n=1, p=probability)
    frame["latent_probability"] = probability.round(6)
    return frame

data = generate_synthetic_data()
display(data.head())
print(data[TARGET_COLUMN].value_counts(normalize=True).round(3))

,age,monthly_income,debt_ratio,credit_utilization,months_employed,delinquencies_12m,credit_history_months,requested_amount,term_months,num_open_accounts,region,channel,customer_segment,defaulted,latent_probability
0,23,15797.85,0.4983,0.3330,36,0,107,105645.55,60,5,Sureste,Movil,PyME,0,0.362422
1,62,12293.14,0.0391,0.3844,49,1,411,244131.01,36,6,Sureste,Sucursal,Premium,0,0.070148
2,55,58917.09,0.3352,0.4552,85,0,424,452726.44,36,2,Occidente,Movil,Masivo,0,0.063145
3,43,106117.91,0.2931,0.3845,91,2,317,437667.92,18,4,Centro,Sucursal,PyME,0,0.062290
4,43,21553.59,0.3003,0.6620,57,0,228,976774.83,36,8,Centro,Digital,Masivo,1,0.818415


defaulted
0    0.841
1    0.159
Name: proportion, dtype: float64


## 3) Entrenamiento, evaluación y registro del modelo

In [16]:
def build_pipeline(random_state: int = 42) -> Pipeline:
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ])
    classifier = LogisticRegression(max_iter=1500, class_weight="balanced", random_state=random_state, solver="lbfgs")
    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier),
    ])

def evaluate(y_true, probability) -> dict[str, float]:
    prediction = (probability >= CONFIG["model_threshold"]).astype(int)
    return {
        "roc_auc": float(roc_auc_score(y_true, probability)),
        "average_precision": float(average_precision_score(y_true, probability)),
        "accuracy": float(accuracy_score(y_true, prediction)),
        "precision": float(precision_score(y_true, prediction, zero_division=0)),
        "recall": float(recall_score(y_true, prediction, zero_division=0)),
        "f1": float(f1_score(y_true, prediction, zero_division=0)),
        "log_loss": float(log_loss(y_true, probability, labels=[0, 1])),
    }

def train_model(frame: pd.DataFrame | None = None, degraded: bool = False, random_state: int = 42) -> dict[str, Any]:
    if frame is None:
        frame = generate_synthetic_data()

    X = frame[FEATURE_COLUMNS].copy()
    y = frame[TARGET_COLUMN].astype(int).copy()

    if degraded:
        rng = np.random.default_rng(random_state + 999)
        flip_mask = rng.random(len(y)) < 0.42
        y.loc[flip_mask] = 1 - y.loc[flip_mask]
        X["monthly_income"] = X["monthly_income"].median()
        X["credit_utilization"] = rng.uniform(0.2, 0.8, size=len(X))

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=random_state
    )

    pipeline = build_pipeline(random_state=random_state)
    pipeline.fit(X_train, y_train)
    probability = pipeline.predict_proba(X_test)[:, 1]
    metrics = evaluate(y_test, probability)

    version = f'{datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")}-{uuid.uuid4().hex[:8]}'
    model_path = MODEL_DIR / f"model-{version}.joblib"
    metadata_path = MODEL_DIR / f"model-{version}.json"

    import joblib
    joblib.dump(pipeline, model_path)

    metadata = {
        "version": version,
        "created_at": utcnow(),
        "algorithm": "LogisticRegression",
        "degraded": degraded,
        "threshold": CONFIG["model_threshold"],
        "feature_columns": FEATURE_COLUMNS,
        "metrics": metrics,
        "model_path": str(model_path),
        "metadata_path": str(metadata_path),
    }

    metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")

    if MLflow_AVAILABLE:
        try:
            mlflow.set_tracking_uri(MLFLOW_DIR.resolve().as_uri())
            mlflow.set_experiment("bank-risk-observability")
            with mlflow.start_run(run_name=f"credit-risk-{version}"):
                mlflow.log_params({
                    "model_version": version,
                    "algorithm": "LogisticRegression",
                    "degraded": degraded,
                    "random_state": random_state,
                    "training_rows": len(X_train),
                    "test_rows": len(X_test),
                })
                mlflow.log_metrics(metrics)
                mlflow.set_tags({
                    "domain": "credit_risk",
                    "environment": "academic_simulation",
                    "organization_context": CONFIG["organization_context"],
                    "quality": "degraded" if degraded else "candidate",
                })
                try:
                    signature_input = X_train.head(20)
                    mlflow.sklearn.log_model(
                        sk_model=pipeline,
                        name="credit_risk_model",
                        input_example=X_train.head(5),
                    )
                except TypeError:
                    mlflow.sklearn.log_model(
                        sk_model=pipeline,
                        artifact_path="credit_risk_model",
                        input_example=X_train.head(5),
                    )
        except Exception as e:
            metadata["mlflow_warning"] = f"MLflow no pudo registrar el experimento: {e}"

    stores["model_registry"].append(metadata)
    return {"model": pipeline, "metadata": metadata, "X_test": X_test, "y_test": y_test, "probability": probability}

trained = train_model(data, degraded=False)
model = trained["model"]
metadata = trained["metadata"]
print(json.dumps(metadata["metrics"], indent=2))

{
  "roc_auc": 0.7804881248933948,
  "average_precision": 0.4491346543022352,
  "accuracy": 0.718,
  "precision": 0.31703703703703706,
  "recall": 0.6750788643533123,
  "f1": 0.4314516129032258,
  "log_loss": 0.556133028297158
}


## 4) Función de inferencia, logging y trazabilidad

In [17]:
def classify_risk(probability: float) -> str:
    if probability >= 0.80:
        return "Crítico"
    if probability >= 0.60:
        return "Alto"
    if probability >= 0.35:
        return "Moderado"
    return "Bajo"

def log_event(level: str, event: str, **kwargs):
    record = {
        "timestamp": utcnow(),
        "level": level,
        "event": event,
        **kwargs,
    }
    stores["logs"].append(record)
    print(json.dumps(record, ensure_ascii=False))

def predict_one(payload: dict[str, Any], actual_default: int | None = None, incident_tag: str | None = None) -> dict[str, Any]:
    trace_id = new_id("trace_")
    start = time.perf_counter()
    try:
        X = pd.DataFrame([payload], columns=FEATURE_COLUMNS)
        probability = float(model.predict_proba(X)[:, 1][0])
        predicted_default = int(probability >= CONFIG["model_threshold"])
        risk_level = classify_risk(probability)
        decision_text = "Revisar con prioridad" if risk_level in {"Alto", "Crítico"} else "Aprobación preliminar"
        latency_ms = (time.perf_counter() - start) * 1000

        result = {
            "trace_id": trace_id,
            "created_at": utcnow(),
            "input_json": payload,
            "probability": probability,
            "predicted_default": predicted_default,
            "risk_level": risk_level,
            "decision_text": decision_text,
            "model_version": metadata["version"],
            "latency_ms": latency_ms,
            "status_code": 200,
            "incident_tag": incident_tag,
            "actual_default": actual_default,
        }
        stores["predictions"].append(result)
        stores["traces"].append({
            "trace_id": trace_id,
            "span": "predict_one",
            "latency_ms": latency_ms,
            "status": "ok",
        })
        log_event("INFO", "prediction_ok", trace_id=trace_id, model_version=metadata["version"], latency_ms=round(latency_ms, 2), risk_level=risk_level)
        return result
    except Exception as e:
        latency_ms = (time.perf_counter() - start) * 1000
        log_event("ERROR", "prediction_failed", trace_id=trace_id, latency_ms=round(latency_ms, 2), error=str(e))
        raise

sample_payload = data[FEATURE_COLUMNS].sample(1, random_state=7).iloc[0].to_dict()
prediction_example = predict_one(sample_payload, actual_default=int(data.sample(1, random_state=7)[TARGET_COLUMN].iloc[0]))
prediction_example

{"timestamp": "2026-07-19T16:13:55.265351+00:00", "level": "INFO", "event": "prediction_ok", "trace_id": "trace_e72783c3d98f", "model_version": "20260719T161355Z-18726a94", "latency_ms": 6.16, "risk_level": "Bajo"}


{'trace_id': 'trace_e72783c3d98f',
 'created_at': '2026-07-19T16:13:55.265351+00:00',
 'input_json': {'age': 71,
  'monthly_income': 36280.9,
  'debt_ratio': 0.3527,
  'credit_utilization': 0.2522,
  'months_employed': 95,
  'delinquencies_12m': 0,
  'credit_history_months': 476,
  'requested_amount': 341225.77,
  'term_months': 18,
  'num_open_accounts': 4,
  'region': 'Occidente',
  'channel': 'Sucursal',
  'customer_segment': 'Masivo'},
 'probability': 0.17041708000323594,
 'predicted_default': 0,
 'risk_level': 'Bajo',
 'decision_text': 'Aprobación preliminar',
 'model_version': '20260719T161355Z-18726a94',
 'latency_ms': 6.157700001494959,
 'status_code': 200,
 'incident_tag': None,
 'actual_default': 0}

## 5) Métricas operativas, modelo y datos

In [18]:
def get_predictions_df() -> pd.DataFrame:
    return pd.DataFrame(stores["predictions"]) if stores["predictions"] else pd.DataFrame()

def compute_operational_metrics(window: int = 200) -> dict[str, float]:
    df = get_predictions_df().tail(window)
    if df.empty:
        return {"count": 0, "error_rate": 0.0, "latency_p50_ms": 0.0, "latency_p95_ms": 0.0, "latency_p99_ms": 0.0, "availability": 1.0}
    latency = df["latency_ms"].astype(float)
    error_rate = float((df["status_code"] >= 500).mean())
    availability = float(1.0 - error_rate)
    return {
        "count": int(len(df)),
        "error_rate": error_rate,
        "latency_p50_ms": float(np.percentile(latency, 50)),
        "latency_p95_ms": float(np.percentile(latency, 95)),
        "latency_p99_ms": float(np.percentile(latency, 99)),
        "availability": availability,
    }

def compute_model_metrics(window: int = 200) -> dict[str, float]:
    df = get_predictions_df().tail(window)
    usable = df.dropna(subset=["actual_default", "probability"])
    if len(usable) < 5:
        return {"roc_auc": np.nan, "precision": np.nan, "recall": np.nan, "f1": np.nan, "log_loss": np.nan}
    y_true = usable["actual_default"].astype(int)
    prob = usable["probability"].astype(float)
    pred = (prob >= CONFIG["model_threshold"]).astype(int)
    return {
        "roc_auc": float(roc_auc_score(y_true, prob)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "log_loss": float(log_loss(y_true, prob, labels=[0, 1])),
    }

def compute_confusion(window: int = 200):
    df = get_predictions_df().tail(window).dropna(subset=["actual_default", "probability"])
    if df.empty:
        return np.array([[0, 0], [0, 0]])
    y_true = df["actual_default"].astype(int)
    y_pred = (df["probability"].astype(float) >= CONFIG["model_threshold"]).astype(int)
    return confusion_matrix(y_true, y_pred, labels=[0, 1])

def reference_profile(df: pd.DataFrame) -> pd.DataFrame:
    return df[FEATURE_COLUMNS].copy()

reference_df = reference_profile(data)
print(reference_df.head())

   age  monthly_income  debt_ratio  credit_utilization  months_employed  \
0   23        15797.85      0.4983              0.3330               36   
1   62        12293.14      0.0391              0.3844               49   
2   55        58917.09      0.3352              0.4552               85   
3   43       106117.91      0.2931              0.3845               91   
4   43        21553.59      0.3003              0.6620               57   

   delinquencies_12m  credit_history_months  requested_amount  term_months  \
0                  0                    107         105645.55           60   
1                  1                    411         244131.01           36   
2                  0                    424         452726.44           36   
3                  2                    317         437667.92           18   
4                  0                    228         976774.83           36   

   num_open_accounts     region   channel customer_segment  
0                  

## 6) Data drift y prediction drift

In [19]:
def population_stability_index(expected: pd.Series, actual: pd.Series, buckets: int = 10) -> float:
    expected = pd.Series(expected).dropna()
    actual = pd.Series(actual).dropna()
    if expected.empty or actual.empty:
        return 0.0

    breaks = np.unique(np.quantile(expected, np.linspace(0, 1, buckets + 1)))
    if len(breaks) < 3:
        return 0.0

    expected_bins = pd.cut(expected, bins=breaks, include_lowest=True, duplicates="drop")
    actual_bins = pd.cut(actual, bins=breaks, include_lowest=True, duplicates="drop")

    expected_dist = expected_bins.value_counts(normalize=True).sort_index()
    actual_dist = actual_bins.value_counts(normalize=True).sort_index()

    idx = expected_dist.index.union(actual_dist.index)
    expected_dist = expected_dist.reindex(idx, fill_value=1e-6)
    actual_dist = actual_dist.reindex(idx, fill_value=1e-6)

    return float(((actual_dist - expected_dist) * np.log(actual_dist / expected_dist)).sum())

def compute_drift_score(reference: pd.DataFrame, current: pd.DataFrame) -> dict[str, Any]:
    score_by_feature = {}
    for column in FEATURE_COLUMNS:
        if column in NUMERIC_FEATURES:
            psi = population_stability_index(reference[column], current[column])
            score_by_feature[column] = float(abs(psi))
        else:
            ref = reference[column].astype(str).value_counts(normalize=True)
            cur = current[column].astype(str).value_counts(normalize=True)
            all_idx = ref.index.union(cur.index)
            ref = ref.reindex(all_idx, fill_value=1e-6)
            cur = cur.reindex(all_idx, fill_value=1e-6)
            score_by_feature[column] = float(((cur - ref) * np.log(cur / ref)).sum())

    drift_score = float(np.mean(list(score_by_feature.values())))
    return {"drift_score": drift_score, "by_feature": score_by_feature}

def compute_prediction_drift(reference_proba: np.ndarray, current_proba: np.ndarray) -> float:
    reference_proba = pd.Series(reference_proba).dropna()
    current_proba = pd.Series(current_proba).dropna()
    if reference_proba.empty or current_proba.empty:
        return 0.0
    return population_stability_index(reference_proba, current_proba)

baseline_proba = model.predict_proba(reference_df.sample(min(500, len(reference_df)), random_state=1))[:, 1]
current_proba = np.asarray([p["probability"] for p in stores["predictions"]]) if stores["predictions"] else np.array([])
drift_snapshot = compute_drift_score(reference_df.sample(500, random_state=1), data.sample(500, random_state=2))
print(json.dumps(drift_snapshot, indent=2))

{
  "drift_score": 0.025593870832714933,
  "by_feature": {
    "age": 0.040175884297230124,
    "monthly_income": 0.05132533294037845,
    "debt_ratio": 0.028810852642238212,
    "credit_utilization": 0.026993166745653246,
    "months_employed": 0.050145455617038925,
    "delinquencies_12m": 0.00014210520889992395,
    "credit_history_months": 0.03948191248355865,
    "requested_amount": 0.036996053275284886,
    "term_months": 0.028922905457081385,
    "num_open_accounts": 0.012326814504946205,
    "region": 0.008453664394217342,
    "channel": 0.005421660380669369,
    "customer_segment": 0.0035245128780973843
  }
}


## 7) SLO, error budgets y motor de alertas

In [20]:
def get_error_budget_remaining(latency_p95_ms: float, error_rate: float, roc_auc: float, drift_score: float) -> float:
    budget = 100.0
    if latency_p95_ms > CONFIG["latency_p95_ms_max"]:
        budget -= min(50, (latency_p95_ms - CONFIG["latency_p95_ms_max"]) / 10)
    if error_rate > CONFIG["error_rate_max"]:
        budget -= min(35, (error_rate - CONFIG["error_rate_max"]) * 1000)
    if not np.isnan(roc_auc) and roc_auc < CONFIG["roc_auc_min"]:
        budget -= min(30, (CONFIG["roc_auc_min"] - roc_auc) * 100)
    if drift_score > CONFIG["drift_score_max"]:
        budget -= min(35, (drift_score - CONFIG["drift_score_max"]) * 100)
    return float(max(0.0, budget))

def create_alert(rule_name: str, severity: str, metric: str, value: float, threshold: float, runbook: str, message: str) -> dict[str, Any]:
    alert = {
        "id": new_id("alert_"),
        "rule_name": rule_name,
        "severity": severity,
        "metric": metric,
        "value": float(value),
        "threshold": float(threshold),
        "status": "open",
        "runbook": runbook,
        "message": message,
        "created_at": utcnow(),
        "updated_at": utcnow(),
    }
    stores["alerts"].append(alert)

    log_event(
        "WARNING" if severity == "warning" else "ERROR",
        "alert_created",
        alert_id=alert["id"],
        rule_name=rule_name,
        severity=severity,
        metric=metric,
        value=float(value),
        threshold=float(threshold),
        runbook=runbook,
        details=message,
    )
    return alert

def resolve_alert(alert_id: str, note: str = "resolved") -> None:
    for alert in stores["alerts"]:
        if alert["id"] == alert_id:
            alert["status"] = "resolved"
            alert["updated_at"] = utcnow()
            alert["resolution_note"] = note
            log_event("INFO", "alert_resolved", alert_id=alert_id, note=note)
            return

def evaluate_alert_rules(metrics: dict[str, float], drift: dict[str, Any]) -> list[dict[str, Any]]:
    alerts = []
    if metrics["latency_p95_ms"] > CONFIG["latency_p95_ms_max"]:
        alerts.append(create_alert(
            "latency_p95_high", "critical", "latency_p95_ms", metrics["latency_p95_ms"],
            CONFIG["latency_p95_ms_max"], "mitigate_latency",
            f"Latencia p95 {metrics['latency_p95_ms']:.1f} ms por encima del SLO."
        ))
    elif metrics["latency_p95_ms"] > 0.8 * CONFIG["latency_p95_ms_max"]:
        alerts.append(create_alert(
            "latency_p95_high", "warning", "latency_p95_ms", metrics["latency_p95_ms"],
            CONFIG["latency_p95_ms_max"], "mitigate_latency",
            f"Latencia p95 en zona de precaucion: {metrics['latency_p95_ms']:.1f} ms."
        ))

    if metrics["error_rate"] > CONFIG["error_rate_max"]:
        alerts.append(create_alert(
            "error_rate_high", "critical", "error_rate", metrics["error_rate"],
            CONFIG["error_rate_max"], "reset_errors",
            f"Tasa de error {metrics['error_rate']:.3%} por encima del umbral."
        ))

    if drift["drift_score"] > CONFIG["drift_score_max"]:
        alerts.append(create_alert(
            "data_drift_high", "critical", "drift_score", drift["drift_score"],
            CONFIG["drift_score_max"], "retrain_model",
            f"Data drift detectado: score {drift['drift_score']:.3f}."
        ))
    elif drift["drift_score"] > 0.35:
        alerts.append(create_alert(
            "data_drift_high", "warning", "drift_score", drift["drift_score"],
            CONFIG["drift_score_max"], "retrain_model",
            f"Deriva moderada: score {drift['drift_score']:.3f}."
        ))

    model_metrics = compute_model_metrics()
    if not np.isnan(model_metrics["roc_auc"]) and model_metrics["roc_auc"] < CONFIG["roc_auc_min"]:
        alerts.append(create_alert(
            "model_roc_auc_low", "critical", "roc_auc", model_metrics["roc_auc"],
            CONFIG["roc_auc_min"], "rollback_model",
            f"ROC-AUC bajo: {model_metrics['roc_auc']:.3f}."
        ))

    budget = get_error_budget_remaining(metrics["latency_p95_ms"], metrics["error_rate"], model_metrics["roc_auc"], drift["drift_score"])
    if budget < 10:
        alerts.append(create_alert(
            "error_budget_low", "critical", "error_budget_remaining", budget,
            10.0, "escalate",
            f"Error budget casi agotado: {budget:.1f}% restante."
        ))
    elif budget < 50:
        alerts.append(create_alert(
            "error_budget_low", "warning", "error_budget_remaining", budget,
            50.0, "escalate",
            f"Error budget reducido: {budget:.1f}% restante."
        ))
    return alerts

current_operational = compute_operational_metrics()
current_model = compute_model_metrics()
current_drift = drift_snapshot
current_budget = get_error_budget_remaining(
    current_operational["latency_p95_ms"],
    current_operational["error_rate"],
    current_model["roc_auc"],
    current_drift["drift_score"],
)

alerts_now = evaluate_alert_rules(current_operational, current_drift)
print("Alertas creadas:", len(alerts_now))
print("Error budget restante:", round(current_budget, 2))

Alertas creadas: 0
Error budget restante: 100.0


## 8) Runbooks: mitigación, reentrenamiento, rollback y escalamiento

In [21]:
def record_runbook(action: str, status: str, details: dict[str, Any], alert_id: str | None = None, incident_id: str | None = None) -> dict[str, Any]:
    runbook = {
        "id": new_id("runbook_"),
        "action": action,
        "status": status,
        "started_at": utcnow(),
        "ended_at": utcnow(),
        "alert_id": alert_id,
        "incident_id": incident_id,
        "details": details,
    }
    stores["runbooks"].append(runbook)
    log_event("INFO", "runbook_executed", action=action, status=status, alert_id=alert_id, incident_id=incident_id, details=details)
    return runbook

def run_mitigate_latency():
    return record_runbook("mitigate_latency", "completed", {
        "action": "se recomienda escalar capacidad simulada y revisar la ruta critica",
        "result": "latencia mitigada a nivel de simulacion",
    })

def run_reset_errors():
    return record_runbook("reset_errors", "completed", {
        "action": "reinicio logico del servicio y limpieza de colas simuladas",
        "result": "errores reducidos",
    })

def run_retrain_model():
    global model, metadata
    retrained = train_model(generate_synthetic_data(mode="drift"), degraded=False, random_state=43)
    model = retrained["model"]
    metadata = retrained["metadata"]
    return record_runbook("retrain_model", "completed", {
        "new_version": metadata["version"],
        "new_metrics": metadata["metrics"],
    })

def run_rollback_model():
    stable_models = [m for m in stores["model_registry"] if not m.get("degraded")]
    if len(stable_models) < 2:
        return record_runbook("rollback_model", "skipped", {"reason": "no hay version estable anterior suficiente"})
    previous = sorted(stable_models, key=lambda x: x["created_at"])[-2]
    model_path = Path(previous["model_path"])
    import joblib
    global model, metadata
    model = joblib.load(model_path)
    metadata = previous
    return record_runbook("rollback_model", "completed", {
        "rolled_back_to": previous["version"],
        "metrics": previous["metrics"],
    })

def run_escalate(alert_id: str | None = None, incident_id: str | None = None):
    return record_runbook("escalate", "completed", {
        "message": "Escalamiento a soporte de nivel superior simulado.",
    }, alert_id=alert_id, incident_id=incident_id)

## 9) Simulador de incidentes

In [22]:
def generate_incident_batch(kind: str, n_samples: int = 150) -> pd.DataFrame:
    if kind == "drift":
        return generate_synthetic_data(n_samples=n_samples, seed=np.random.randint(1, 1_000_000), mode="drift")
    if kind == "severe_drift":
        return generate_synthetic_data(n_samples=n_samples, seed=np.random.randint(1, 1_000_000), mode="severe_drift")
    return generate_synthetic_data(n_samples=n_samples, seed=np.random.randint(1, 1_000_000), mode="normal")

def simulate_traffic(batch: pd.DataFrame, incident_tag: str | None = None, inject_latency_ms: float = 0.0, force_error_rate: float = 0.0) -> pd.DataFrame:
    results = []
    rng = np.random.default_rng(2025)
    for _, row in batch.iterrows():
        payload = row[FEATURE_COLUMNS].to_dict()
        y_true = int(row[TARGET_COLUMN])
        if force_error_rate and rng.random() < force_error_rate:
            start = time.perf_counter()
            time.sleep(inject_latency_ms / 1000.0)
            latency_ms = (time.perf_counter() - start) * 1000
            record = {
                "trace_id": new_id("trace_"),
                "created_at": utcnow(),
                "input_json": payload,
                "probability": np.nan,
                "predicted_default": np.nan,
                "risk_level": "Error",
                "decision_text": "Fallo de servicio",
                "model_version": metadata["version"],
                "latency_ms": latency_ms,
                "status_code": 500,
                "incident_tag": incident_tag,
                "actual_default": y_true,
            }
            stores["predictions"].append(record)
            results.append(record)
        else:
            if inject_latency_ms > 0:
                time.sleep(inject_latency_ms / 1000.0)
            record = predict_one(payload, actual_default=y_true, incident_tag=incident_tag)
            results.append(record)
    return pd.DataFrame(results)

def open_incident(kind: str, details: dict[str, Any]) -> dict[str, Any]:
    incident = {
        "id": new_id("incident_"),
        "kind": kind,
        "status": "open",
        "started_at": utcnow(),
        "ended_at": None,
        "details": details,
    }
    stores["incidents"].append(incident)
    log_event("ERROR", "incident_opened", **incident)
    return incident

def close_incident(incident_id: str, note: str = "resolved") -> None:
    for incident in stores["incidents"]:
        if incident["id"] == incident_id:
            incident["status"] = "closed"
            incident["ended_at"] = utcnow()
            incident["resolution_note"] = note
            log_event("INFO", "incident_closed", incident_id=incident_id, note=note)
            return

## 10) Ejecutar tráfico normal e incidentes simulados

In [23]:
# Tráfico normal
normal_batch = generate_incident_batch("normal", n_samples=120)
normal_results = simulate_traffic(normal_batch, incident_tag="normal")

# Incidente 1: data drift
drift_incident = open_incident(
    "data_drift",
    {"description": "Cambio sintético en distribuciones de entrada."}
)

drift_batch = generate_incident_batch("drift", n_samples=160)
drift_results = simulate_traffic(drift_batch, incident_tag="data_drift")

sample_size_ref = min(500, len(reference_df))
sample_size_drift = min(500, len(drift_batch))

drift_after = compute_drift_score(
    reference_df.sample(sample_size_ref, random_state=3),
    drift_batch[FEATURE_COLUMNS].sample(sample_size_drift, random_state=4)
)

drift_metrics_after = compute_operational_metrics()
alerts_drift = evaluate_alert_rules(drift_metrics_after, drift_after)
run_retrain = run_retrain_model()

close_incident(
    drift_incident["id"],
    "se ejecuto runbook de reentrenamiento"
)

# Incidente 2: incremento de latencia y errores
latency_incident = open_incident(
    "latency",
    {"description": "Aumento sintético de latencia y errores."}
)

latency_batch = generate_incident_batch("normal", n_samples=100)
latency_results = simulate_traffic(
    latency_batch,
    incident_tag="latency",
    inject_latency_ms=35,
    force_error_rate=0.18
)

latency_metrics = compute_operational_metrics()
latency_model_metrics = compute_model_metrics()
latency_drift = compute_drift_score(
    reference_df.sample(min(500, len(reference_df)), random_state=5),
    data.sample(min(500, len(data)), random_state=6)
)

alerts_latency = evaluate_alert_rules(latency_metrics, latency_drift)
run_latency = run_mitigate_latency()
run_errors = run_reset_errors()
run_escalation = run_escalate(incident_id=latency_incident["id"])

close_incident(
    latency_incident["id"],
    "se aplicaron runbooks de latencia y errores"
)

print("Escenarios ejecutados.")
print("Total predicciones:", len(stores["predictions"]))
print("Total alertas:", len(stores["alerts"]))
print("Total incidentes:", len(stores["incidents"]))


{"timestamp": "2026-07-19T16:13:55.406788+00:00", "level": "INFO", "event": "prediction_ok", "trace_id": "trace_646cd720c335", "model_version": "20260719T161355Z-18726a94", "latency_ms": 5.6, "risk_level": "Bajo"}
{"timestamp": "2026-07-19T16:13:55.415882+00:00", "level": "INFO", "event": "prediction_ok", "trace_id": "trace_2e5922422734", "model_version": "20260719T161355Z-18726a94", "latency_ms": 6.8, "risk_level": "Bajo"}
{"timestamp": "2026-07-19T16:13:55.424118+00:00", "level": "INFO", "event": "prediction_ok", "trace_id": "trace_a6309658be22", "model_version": "20260719T161355Z-18726a94", "latency_ms": 8.8, "risk_level": "Alto"}
{"timestamp": "2026-07-19T16:13:55.434672+00:00", "level": "INFO", "event": "prediction_ok", "trace_id": "trace_75ef4f6420b4", "model_version": "20260719T161355Z-18726a94", "latency_ms": 8.55, "risk_level": "Moderado"}
{"timestamp": "2026-07-19T16:13:55.439678+00:00", "level": "INFO", "event": "prediction_ok", "trace_id": "trace_7aa6a5feef73", "model_versi

C:\Users\MI42678\AppData\Roaming\Python\Python312\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\Users\MI42678\AppData\Roaming\Python\Python312\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


{"timestamp": "2026-07-19T16:13:56.568894+00:00", "level": "INFO", "event": "prediction_ok", "trace_id": "trace_12f77ca80f0a", "model_version": "20260719T161356Z-71b624fc", "latency_ms": 7.42, "risk_level": "Bajo"}
{"timestamp": "2026-07-19T16:13:56.612071+00:00", "level": "INFO", "event": "prediction_ok", "trace_id": "trace_c8dc2c6eb595", "model_version": "20260719T161356Z-71b624fc", "latency_ms": 4.55, "risk_level": "Bajo"}
{"timestamp": "2026-07-19T16:13:56.652903+00:00", "level": "INFO", "event": "prediction_ok", "trace_id": "trace_739b3e462224", "model_version": "20260719T161356Z-71b624fc", "latency_ms": 6.23, "risk_level": "Bajo"}
{"timestamp": "2026-07-19T16:13:56.736377+00:00", "level": "INFO", "event": "prediction_ok", "trace_id": "trace_cc2a81eb5ddb", "model_version": "20260719T161356Z-71b624fc", "latency_ms": 9.46, "risk_level": "Bajo"}
{"timestamp": "2026-07-19T16:13:56.776658+00:00", "level": "INFO", "event": "prediction_ok", "trace_id": "trace_a18827a68970", "model_versio

## 11) Panel de resumen ejecutivo

In [24]:
def latest_state() -> dict[str, Any]:
    op = compute_operational_metrics()
    mm = compute_model_metrics()
    dr = compute_drift_score(reference_df.sample(500, random_state=7), data.sample(500, random_state=8))
    budget = get_error_budget_remaining(op["latency_p95_ms"], op["error_rate"], mm["roc_auc"], dr["drift_score"])
    return {
        "operational": op,
        "model": mm,
        "drift": dr,
        "error_budget_remaining": budget,
        "alerts_open": sum(1 for a in stores["alerts"] if a["status"] == "open"),
        "incidents_open": sum(1 for i in stores["incidents"] if i["status"] == "open"),
        "model_version": metadata["version"],
    }

state = latest_state()
state

{'operational': {'count': 200,
  'error_rate': 0.065,
  'latency_p50_ms': 4.316399994422682,
  'latency_p95_ms': 35.262515013164375,
  'latency_p99_ms': 35.67176301730797,
  'availability': 0.935},
 'model': {'roc_auc': 0.8019704433497538,
  'precision': 0.3974358974358974,
  'recall': 0.7380952380952381,
  'f1': 0.5166666666666667,
  'log_loss': 0.5490491642607253},
 'drift': {'drift_score': 0.0251933919669037,
  'by_feature': {'age': 0.025977330664891423,
   'monthly_income': 0.03657543682406014,
   'debt_ratio': 0.015478381612119665,
   'credit_utilization': 0.019161646524078067,
   'months_employed': 0.05481548707961319,
   'delinquencies_12m': 0.00013320501071684542,
   'credit_history_months': 0.04245025460609888,
   'requested_amount': 0.05837605017847017,
   'term_months': 0.0485516030289759,
   'num_open_accounts': 0.005997046305567945,
   'region': 0.007583030651261856,
   'channel': 0.008316413516921247,
   'customer_segment': 0.004098209566972692}},
 'error_budget_remaining

## 12) Dashboard interactivo dentro del notebook

In [25]:
def build_dashboard():
    state = latest_state()
    op = state["operational"]
    mm = state["model"]
    dr = state["drift"]

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=("Latencia por predicción", "Distribucion de riesgo", "Data drift por variable", "Matriz de confusion"),
        specs=[[{"type": "xy"}, {"type": "domain"}], [{"type": "bar"}, {"type": "heatmap"}]],
    )

    pred_df = get_predictions_df().tail(300).copy()
    if not pred_df.empty:
        fig.add_trace(go.Scatter(y=pred_df["latency_ms"], mode="lines", name="latency_ms"), row=1, col=1)
        if "risk_level" in pred_df.columns:
            risk_counts = pred_df["risk_level"].astype(str).value_counts()
            fig.add_trace(go.Pie(labels=risk_counts.index, values=risk_counts.values, name="Riesgo"), row=1, col=2)

    drift_items = pd.Series(dr["by_feature"]).sort_values(ascending=False)
    fig.add_trace(go.Bar(x=drift_items.index, y=drift_items.values, name="drift"), row=2, col=1)

    cm = compute_confusion()
    fig.add_trace(go.Heatmap(z=cm, x=["Pred 0", "Pred 1"], y=["Real 0", "Real 1"], showscale=True), row=2, col=2)

    fig.update_layout(height=800, title_text="Dashboard operativo de riesgo bancario", showlegend=False)

    summary = pd.DataFrame([{
        "Metric": "Availability",
        "Value": round(op["availability"], 4),
    }, {
        "Metric": "Latency p95 (ms)",
        "Value": round(op["latency_p95_ms"], 2),
    }, {
        "Metric": "Error rate",
        "Value": round(op["error_rate"], 4),
    }, {
        "Metric": "ROC-AUC",
        "Value": None if np.isnan(mm["roc_auc"]) else round(mm["roc_auc"], 4),
    }, {
        "Metric": "Drift score",
        "Value": round(dr["drift_score"], 4),
    }, {
        "Metric": "Error budget restante",
        "Value": round(state["error_budget_remaining"], 2),
    }, {
        "Metric": "Alertas abiertas",
        "Value": state["alerts_open"],
    }, {
        "Metric": "Incidentes abiertos",
        "Value": state["incidents_open"],
    }])

    display(summary)
    display(fig)

    alerts_df = pd.DataFrame(stores["alerts"]).tail(10)
    incidents_df = pd.DataFrame(stores["incidents"]).tail(10)
    runbooks_df = pd.DataFrame(stores["runbooks"]).tail(10)
    print("Alertas recientes:")
    display(alerts_df if not alerts_df.empty else pd.DataFrame([{"message": "Sin alertas"}]))
    print("Incidentes recientes:")
    display(incidents_df if not incidents_df.empty else pd.DataFrame([{"message": "Sin incidentes"}]))
    print("Runbooks recientes:")
    display(runbooks_df if not runbooks_df.empty else pd.DataFrame([{"message": "Sin runbooks"}]))

build_dashboard()

,Metric,Value
0,Availability,0.9350
1,Latency p95 (ms),35.2600
2,Error rate,0.0650
3,ROC-AUC,0.8020
4,Drift score,0.0252
5,Error budget restante,65.0000
6,Alertas abiertas,2.0000
7,Incidentes abiertos,0.0000


Alertas recientes:


,id,rule_name,severity,metric,value,threshold,status,runbook,message,created_at,updated_at
0,alert_d0a1b6542c38,data_drift_high,critical,drift_score,inf,0.50,open,retrain_model,Data drift detectado: score inf.,2026-07-19T16:13:56.383245+00:00,2026-07-19T16:13:56.383245+00:00
1,alert_574f8b2ecd91,error_rate_high,critical,error_rate,0.065,0.01,open,reset_errors,Tasa de error 6.500% por encima del umbral.,2026-07-19T16:14:00.846095+00:00,2026-07-19T16:14:00.846095+00:00


Incidentes recientes:


,id,kind,status,started_at,ended_at,details,resolution_note
0,incident_43cf080bc439,data_drift,closed,2026-07-19T16:13:55.863951+00:00,2026-07-19T16:13:56.432317+00:00,{'description': 'Cambio sintético en distribuc...,se ejecuto runbook de reentrenamiento
1,incident_535cede2876d,latency,closed,2026-07-19T16:13:56.432317+00:00,2026-07-19T16:14:00.855991+00:00,{'description': 'Aumento sintético de latencia...,se aplicaron runbooks de latencia y errores


Runbooks recientes:


,id,action,status,started_at,ended_at,alert_id,incident_id,details
0,runbook_710f5899946b,retrain_model,completed,2026-07-19T16:13:56.432317+00:00,2026-07-19T16:13:56.432317+00:00,None,None,"{'new_version': '20260719T161356Z-71b624fc', '..."
1,runbook_00f1ec01990c,mitigate_latency,completed,2026-07-19T16:14:00.855991+00:00,2026-07-19T16:14:00.855991+00:00,None,None,{'action': 'se recomienda escalar capacidad si...
2,runbook_31afb128fb77,reset_errors,completed,2026-07-19T16:14:00.855991+00:00,2026-07-19T16:14:00.855991+00:00,None,None,{'action': 'reinicio logico del servicio y lim...
3,runbook_b06056ad8888,escalate,completed,2026-07-19T16:14:00.855991+00:00,2026-07-19T16:14:00.855991+00:00,None,incident_535cede2876d,{'message': 'Escalamiento a soporte de nivel s...


## 13) API opcional embebida en el notebook

In [26]:
if FASTAPI_AVAILABLE:
    class LoanApplication(BaseModel):
        age: int = Field(ge=18, le=80)
        monthly_income: float = Field(gt=0, le=1_000_000)
        debt_ratio: float = Field(ge=0, le=2.0)
        credit_utilization: float = Field(ge=0, le=1.5)
        months_employed: int = Field(ge=0, le=600)
        delinquencies_12m: int = Field(ge=0, le=24)
        credit_history_months: int = Field(ge=0, le=720)
        requested_amount: float = Field(ge=1_000, le=10_000_000)
        term_months: int = Field(ge=6, le=120)
        num_open_accounts: int = Field(ge=0, le=50)
        region: Literal["Norte", "Centro", "Occidente", "Sureste"]
        channel: Literal["Sucursal", "Digital", "Movil"]
        customer_segment: Literal["Masivo", "Premium", "PyME"]

    app = FastAPI(title="Bank Risk Observability API", version="1.0.0")

    @app.get("/health")
    def health():
        return {"status": "ok", "model_version": metadata["version"]}

    @app.get("/metrics")
    def metrics():
        op = compute_operational_metrics()
        mm = compute_model_metrics()
        dr = compute_drift_score(reference_df.sample(500, random_state=9), data.sample(500, random_state=10))
        return {
            "operational": op,
            "model": mm,
            "drift": dr,
            "error_budget_remaining": get_error_budget_remaining(op["latency_p95_ms"], op["error_rate"], mm["roc_auc"], dr["drift_score"]),
        }

    @app.post("/predict")
    def predict(payload: LoanApplication):
        result = predict_one(payload.model_dump())
        return JSONResponse(result)

    print("FastAPI app creada en la variable `app`.")
else:
    app = None
    print("FastAPI no está disponible en este entorno.")

FastAPI app creada en la variable `app`.


## 14) Exportar estados y guardar resultados dentro del notebook

In [27]:
summary_tables = {
    "predictions": pd.DataFrame(stores["predictions"]).tail(20),
    "alerts": pd.DataFrame(stores["alerts"]).tail(20),
    "incidents": pd.DataFrame(stores["incidents"]).tail(20),
    "runbooks": pd.DataFrame(stores["runbooks"]).tail(20),
    "models": pd.DataFrame(stores["model_registry"]).tail(10),
}

for name, df in summary_tables.items():
    print(f"--- {name} ---")
    display(df if not df.empty else pd.DataFrame([{"message": "sin datos"}]))

--- predictions ---


,trace_id,created_at,input_json,probability,predicted_default,risk_level,decision_text,model_version,latency_ms,status_code,incident_tag,actual_default
361,trace_112b70a42c78,2026-07-19T16:13:59.932189+00:00,"{'age': 28, 'monthly_income': 33021.98, 'debt_...",0.096356,0.0,Bajo,Aprobación preliminar,20260719T161356Z-71b624fc,8.6393,200,latency,0
362,trace_3dc5d5a84f07,2026-07-19T16:13:59.970615+00:00,"{'age': 41, 'monthly_income': 42716.31, 'debt_...",0.107367,0.0,Bajo,Aprobación preliminar,20260719T161356Z-71b624fc,7.5658,200,latency,0
363,trace_f3926b3ca690,2026-07-19T16:14:00.023048+00:00,"{'age': 55, 'monthly_income': 31310.75, 'debt_...",0.044417,0.0,Bajo,Aprobación preliminar,20260719T161356Z-71b624fc,9.8773,200,latency,0
364,trace_80eca0e97eda,2026-07-19T16:14:00.066939+00:00,"{'age': 40, 'monthly_income': 20198.47, 'debt_...",0.251250,0.0,Bajo,Aprobación preliminar,20260719T161356Z-71b624fc,8.8420,200,latency,0
365,trace_9396ce8623c8,2026-07-19T16:14:00.111399+00:00,"{'age': 42, 'monthly_income': 23631.87, 'debt_...",0.411192,0.0,Moderado,Aprobación preliminar,20260719T161356Z-71b624fc,4.8657,200,latency,0
366,trace_f70c15405e22,2026-07-19T16:14:00.149522+00:00,"{'age': 66, 'monthly_income': 16972.74, 'debt_...",0.195267,0.0,Bajo,Aprobación preliminar,20260719T161356Z-71b624fc,6.0990,200,latency,0
367,trace_55422d317fde,2026-07-19T16:14:00.202066+00:00,"{'age': 68, 'monthly_income': 42337.65, 'debt_...",0.053359,0.0,Bajo,Aprobación preliminar,20260719T161356Z-71b624fc,8.6089,200,latency,0
368,trace_6b18e6269c10,2026-07-19T16:14:00.245347+00:00,"{'age': 25, 'monthly_income': 9372.08, 'debt_r...",0.666248,1.0,Alto,Revisar con prioridad,20260719T161356Z-71b624fc,5.4512,200,latency,0
369,trace_1c48d6764f2e,2026-07-19T16:14:00.283494+00:00,"{'age': 25, 'monthly_income': 19171.13, 'debt_...",0.195426,0.0,Bajo,Aprobación preliminar,20260719T161356Z-71b624fc,7.7891,200,latency,0
370,trace_1266833e7643,2026-07-19T16:14:00.329391+00:00,"{'age': 50, 'monthly_income': 19574.5, 'debt_r...",0.128931,0.0,Bajo,Aprobación preliminar,20260719T161356Z-71b624fc,5.7684,200,latency,0


--- alerts ---


,id,rule_name,severity,metric,value,threshold,status,runbook,message,created_at,updated_at
0,alert_d0a1b6542c38,data_drift_high,critical,drift_score,inf,0.50,open,retrain_model,Data drift detectado: score inf.,2026-07-19T16:13:56.383245+00:00,2026-07-19T16:13:56.383245+00:00
1,alert_574f8b2ecd91,error_rate_high,critical,error_rate,0.065,0.01,open,reset_errors,Tasa de error 6.500% por encima del umbral.,2026-07-19T16:14:00.846095+00:00,2026-07-19T16:14:00.846095+00:00


--- incidents ---


,id,kind,status,started_at,ended_at,details,resolution_note
0,incident_43cf080bc439,data_drift,closed,2026-07-19T16:13:55.863951+00:00,2026-07-19T16:13:56.432317+00:00,{'description': 'Cambio sintético en distribuc...,se ejecuto runbook de reentrenamiento
1,incident_535cede2876d,latency,closed,2026-07-19T16:13:56.432317+00:00,2026-07-19T16:14:00.855991+00:00,{'description': 'Aumento sintético de latencia...,se aplicaron runbooks de latencia y errores


--- runbooks ---


,id,action,status,started_at,ended_at,alert_id,incident_id,details
0,runbook_710f5899946b,retrain_model,completed,2026-07-19T16:13:56.432317+00:00,2026-07-19T16:13:56.432317+00:00,None,None,"{'new_version': '20260719T161356Z-71b624fc', '..."
1,runbook_00f1ec01990c,mitigate_latency,completed,2026-07-19T16:14:00.855991+00:00,2026-07-19T16:14:00.855991+00:00,None,None,{'action': 'se recomienda escalar capacidad si...
2,runbook_31afb128fb77,reset_errors,completed,2026-07-19T16:14:00.855991+00:00,2026-07-19T16:14:00.855991+00:00,None,None,{'action': 'reinicio logico del servicio y lim...
3,runbook_b06056ad8888,escalate,completed,2026-07-19T16:14:00.855991+00:00,2026-07-19T16:14:00.855991+00:00,None,incident_535cede2876d,{'message': 'Escalamiento a soporte de nivel s...


--- models ---


,version,created_at,algorithm,degraded,threshold,feature_columns,metrics,model_path,metadata_path,mlflow_warning
0,20260719T161355Z-18726a94,2026-07-19T16:13:55.238920+00:00,LogisticRegression,False,0.5,"[age, monthly_income, debt_ratio, credit_utili...","{'roc_auc': 0.7804881248933948, 'average_preci...",C:\Users\MI42678\AppData\Local\Temp\bank_risk_...,C:\Users\MI42678\AppData\Local\Temp\bank_risk_...,MLflow no pudo registrar el experimento: The f...
1,20260719T161356Z-71b624fc,2026-07-19T16:13:56.432317+00:00,LogisticRegression,False,0.5,"[age, monthly_income, debt_ratio, credit_utili...","{'roc_auc': 0.7833407310355892, 'average_preci...",C:\Users\MI42678\AppData\Local\Temp\bank_risk_...,C:\Users\MI42678\AppData\Local\Temp\bank_risk_...,MLflow no pudo registrar el experimento: The f...


In [30]:
from IPython.display import HTML, display, clear_output

def show_kpi_cards(op, mm, drift, error_budget_remaining):
    html = f"""
    <style>
      .kpi-grid {{
        display: grid;
        grid-template-columns: repeat(3, minmax(180px, 1fr));
        gap: 12px;
        margin: 10px 0 18px 0;
      }}
      .kpi-card {{
        background: #ffffff;
        border: 1px solid #e6eaf2;
        border-radius: 16px;
        padding: 14px 16px;
        box-shadow: 0 2px 10px rgba(16, 24, 40, 0.05);
      }}
      .kpi-label {{
        font-size: 13px;
        color: #6b7280;
        margin-bottom: 6px;
      }}
      .kpi-value {{
        font-size: 28px;
        font-weight: 700;
        color: #1f3b66;
        line-height: 1.05;
      }}
      .kpi-sub {{
        font-size: 12px;
        color: #8a94a6;
        margin-top: 6px;
      }}
    </style>
    <div class="kpi-grid">
      <div class="kpi-card"><div class="kpi-label">Disponibilidad</div><div class="kpi-value">{op["availability"]*100:.2f}%</div><div class="kpi-sub">Objetivo: {CONFIG["availability_target"]*100:.2f}%</div></div>
      <div class="kpi-card"><div class="kpi-label">Latencia p95</div><div class="kpi-value">{op["latency_p95_ms"]:.1f} ms</div><div class="kpi-sub">Máximo: {CONFIG["latency_p95_ms_max"]} ms</div></div>
      <div class="kpi-card"><div class="kpi-label">Error rate</div><div class="kpi-value">{op["error_rate"]*100:.2f}%</div><div class="kpi-sub">Máximo: {CONFIG["error_rate_max"]*100:.2f}%</div></div>
      <div class="kpi-card"><div class="kpi-label">ROC-AUC</div><div class="kpi-value">{0 if np.isnan(mm["roc_auc"]) else mm["roc_auc"]:.3f}</div><div class="kpi-sub">Mínimo: {CONFIG["roc_auc_min"]:.2f}</div></div>
      <div class="kpi-card"><div class="kpi-label">Drift score</div><div class="kpi-value">{drift["drift_score"]:.3f}</div><div class="kpi-sub">Umbral: {CONFIG["drift_score_max"]:.2f}</div></div>
      <div class="kpi-card"><div class="kpi-label">Error budget</div><div class="kpi-value">{error_budget_remaining:.1f}%</div><div class="kpi-sub">Restante en la ventana actual</div></div>
    </div>
    """
    display(HTML(html))


def render_advanced_dashboard(window: int = 300, incident_filter: str = "All"):
    pred_df = get_predictions_df().copy()
    if pred_df.empty:
        display(pd.DataFrame([{"message": "Aun no hay predicciones registradas."}]))
        return

    pred_df["created_at"] = pd.to_datetime(pred_df["created_at"], errors="coerce", utc=True)
    pred_df = pred_df.sort_values("created_at")

    if incident_filter != "All" and "incident_tag" in pred_df.columns:
        pred_df = pred_df[pred_df["incident_tag"].astype(str) == incident_filter]

    pred_df = pred_df.tail(window).copy()
    if pred_df.empty:
        display(pd.DataFrame([{"message": "No hay datos para la ventana seleccionada."}]))
        return

    op = compute_operational_metrics(window=window)
    mm = compute_model_metrics(window=window)

    usable = pred_df.dropna(subset=["actual_default", "probability"]).copy()
    if not usable.empty:
        y_true = usable["actual_default"].astype(int).to_numpy()
        y_prob = usable["probability"].astype(float).to_numpy()
    else:
        y_true = np.array([])
        y_prob = np.array([])

    reference_sample = reference_df.sample(min(500, len(reference_df)), random_state=11)

    input_series = pred_df.get("input_json", pd.Series([{}] * len(pred_df), index=pred_df.index))
    input_series = input_series.apply(lambda x: x if isinstance(x, dict) else {})
    current_feature_sample = pd.json_normalize(input_series)
    current_feature_sample = current_feature_sample.reindex(columns=FEATURE_COLUMNS)

    if len(current_feature_sample) > 0:
        current_sample = current_feature_sample.sample(
            min(500, len(current_feature_sample)),
            random_state=12
        )
    else:
        current_sample = current_feature_sample

    drift = compute_drift_score(reference_sample, current_sample)

    error_budget_remaining = get_error_budget_remaining(
        op["latency_p95_ms"],
        op["error_rate"],
        mm["roc_auc"] if not np.isnan(mm["roc_auc"]) else 0.0,
        drift["drift_score"],
    )

    show_kpi_cards(op, mm, drift, error_budget_remaining)

    summary = pd.DataFrame([{
        "Metrica": "Predicciones",
        "Valor": int(op["count"]),
    }, {
        "Metrica": "Disponibilidad",
        "Valor": round(op["availability"] * 100, 2),
    }, {
        "Metrica": "Latencia p95 (ms)",
        "Valor": round(op["latency_p95_ms"], 2),
    }, {
        "Metrica": "Error rate (%)",
        "Valor": round(op["error_rate"] * 100, 2),
    }, {
        "Metrica": "ROC-AUC",
        "Valor": None if np.isnan(mm["roc_auc"]) else round(mm["roc_auc"], 3),
    }, {
        "Metrica": "Drift score",
        "Valor": round(drift["drift_score"], 3),
    }, {
        "Metrica": "Error budget (%)",
        "Valor": round(error_budget_remaining, 2),
    }, {
        "Metrica": "Alertas abiertas",
        "Valor": sum(1 for a in stores["alerts"] if a["status"] == "open"),
    }, {
        "Metrica": "Incidentes abiertos",
        "Valor": sum(1 for i in stores["incidents"] if i["status"] == "open"),
    }])

    display(
        summary.style
        .hide(axis="index")
        .set_table_styles([
            {"selector": "th", "props": [("text-align", "center"), ("background-color", "#f3f6fb")]},
            {"selector": "td", "props": [("text-align", "center")]},
            {"selector": "table", "props": [("width", "50%"), ("margin-bottom", "12px")]},
        ])
    )

    # Serie temporal
    timeline = pred_df[["created_at", "latency_ms", "status_code"]].copy()
    timeline["latency_ms"] = pd.to_numeric(timeline["latency_ms"], errors="coerce")
    timeline["error_flag"] = (timeline["status_code"] >= 500).astype(int)
    timeline = timeline.dropna(subset=["created_at", "latency_ms"]).sort_values("created_at")

    roll_window = min(30, max(5, len(timeline)))
    timeline["rolling_p95"] = timeline["latency_ms"].rolling(window=roll_window, min_periods=1).quantile(0.95)
    timeline["rolling_error_rate"] = timeline["error_flag"].rolling(window=roll_window, min_periods=1).mean()

    ts_fig = make_subplots(specs=[[{"secondary_y": True}]])
    ts_fig.add_trace(go.Scatter(x=timeline["created_at"], y=timeline["latency_ms"], mode="lines", name="Latencia (ms)"), secondary_y=False)
    ts_fig.add_trace(go.Scatter(x=timeline["created_at"], y=timeline["rolling_p95"], mode="lines", name="Rolling p95", line=dict(dash="dot")), secondary_y=False)
    ts_fig.add_trace(go.Scatter(x=timeline["created_at"], y=timeline["rolling_error_rate"] * 100, mode="lines", name="Rolling error %"), secondary_y=True)
    ts_fig.add_trace(go.Scatter(x=timeline["created_at"], y=np.repeat(CONFIG["latency_p95_ms_max"], len(timeline)), mode="lines", name="SLO latencia", line=dict(dash="dash")), secondary_y=False)
    ts_fig.update_layout(
        title="Serie temporal operativa",
        height=460,
        margin=dict(l=40, r=30, t=70, b=50),
        title_x=0.02,
        template="plotly_white",
        legend=dict(orientation="h")
    )
    ts_fig.update_yaxes(title_text="Latencia (ms)", secondary_y=False)
    ts_fig.update_yaxes(title_text="Error rate (%)", secondary_y=True)
    display(ts_fig)

    # ROC / PR / Confusion
    roc_fig = go.Figure()
    pr_fig = go.Figure()
    if len(usable) >= 10 and len(np.unique(y_true)) > 1:
        from sklearn.metrics import roc_curve, precision_recall_curve, auc
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        precision, recall, _ = precision_recall_curve(y_true, y_prob)
        roc_auc_val = auc(fpr, tpr)
        pr_auc_val = auc(recall, precision)

        roc_fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name=f"ROC AUC = {roc_auc_val:.3f}"))
        roc_fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Azar", line=dict(dash="dash")))
        roc_fig.update_layout(title="Curva ROC", xaxis_title="FPR", yaxis_title="TPR", height=360, template="plotly_white", margin=dict(l=40, r=30, t=70, b=50), title_x=0.02)

        pr_fig.add_trace(go.Scatter(x=recall, y=precision, mode="lines", name=f"PR AUC = {pr_auc_val:.3f}"))
        pr_fig.update_layout(title="Curva Precision-Recall", xaxis_title="Recall", yaxis_title="Precision", height=360, template="plotly_white", margin=dict(l=40, r=30, t=70, b=50), title_x=0.02)
    else:
        roc_fig.add_annotation(text="No hay datos suficientes para ROC", x=0.5, y=0.5, showarrow=False)
        pr_fig.add_annotation(text="No hay datos suficientes para PR", x=0.5, y=0.5, showarrow=False)
        roc_fig.update_layout(height=360, template="plotly_white")
        pr_fig.update_layout(height=360, template="plotly_white")

    cm = compute_confusion(window=window)
    cm_fig = go.Figure(data=go.Heatmap(
        z=cm,
        x=["Pred 0", "Pred 1"],
        y=["Real 0", "Real 1"],
        text=cm,
        texttemplate="%{text}",
        colorscale="Blues",
        showscale=True
    ))
    cm_fig.update_layout(title="Matriz de confusión", height=360, template="plotly_white", margin=dict(l=40, r=30, t=70, b=50), title_x=0.02)

    top_row = make_subplots(
        rows=1, cols=3,
        subplot_titles=("ROC", "Precision-Recall", "Matriz de confusión"),
        specs=[[{"type": "xy"}, {"type": "xy"}, {"type": "heatmap"}]]
    )
    for tr in roc_fig.data:
        top_row.add_trace(tr, row=1, col=1)
    for tr in pr_fig.data:
        top_row.add_trace(tr, row=1, col=2)
    for tr in cm_fig.data:
        top_row.add_trace(tr, row=1, col=3)

    top_row.update_layout(
        height=430,
        margin=dict(l=40, r=30, t=80, b=50),
        title="Desempeño del modelo",
        title_x=0.02,
        template="plotly_white"
    )
    display(top_row)

    # Drift + distribución
    drift_series = pd.Series(drift["by_feature"]).sort_values(ascending=False)
    drift_fig = go.Figure()
    drift_fig.add_trace(go.Bar(
        x=drift_series.index,
        y=drift_series.values,
        text=drift_series.values.round(3),
        textposition="outside",
        name="Drift"
    ))
    drift_fig.add_hline(y=CONFIG["drift_score_max"], line_dash="dash", line_color="red")
    drift_fig.update_layout(
        title="Drift por variable",
        xaxis_title="Variable",
        yaxis_title="Score",
        height=360,
        template="plotly_white",
        margin=dict(l=40, r=30, t=70, b=50),
        title_x=0.02
    )
    drift_fig.update_xaxes(tickangle=-35)

    dist_fig = go.Figure()
    dist_fig.add_trace(go.Histogram(
        x=usable["probability"] if not usable.empty else pred_df["probability"].dropna(),
        nbinsx=25,
        name="Probabilidad",
        opacity=0.85
    ))
    dist_fig.update_layout(
        title="Distribución de probabilidad",
        height=360,
        template="plotly_white",
        margin=dict(l=40, r=30, t=70, b=50),
        title_x=0.02
    )

    bottom_row = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Drift por variable", "Distribución de probabilidad"),
        specs=[[{"type": "bar"}, {"type": "histogram"}]]
    )
    for tr in drift_fig.data:
        bottom_row.add_trace(tr, row=1, col=1)
    for tr in dist_fig.data:
        bottom_row.add_trace(tr, row=1, col=2)

    bottom_row.update_layout(
        height=430,
        margin=dict(l=40, r=30, t=80, b=50),
        title="Drift y distribución",
        title_x=0.02,
        template="plotly_white"
    )
    display(bottom_row)

    # Tablas finales
    alerts_df = pd.DataFrame(stores["alerts"]).copy()
    if not alerts_df.empty:
        alerts_df["created_at"] = pd.to_datetime(alerts_df["created_at"], errors="coerce", utc=True)
        alerts_df = alerts_df.sort_values("created_at", ascending=False).head(10)
    else:
        alerts_df = pd.DataFrame([{"message": "Sin alertas"}])

    incidents_df = pd.DataFrame(stores["incidents"]).copy()
    if not incidents_df.empty:
        incidents_df["started_at"] = pd.to_datetime(incidents_df["started_at"], errors="coerce", utc=True)
        incidents_df = incidents_df.sort_values("started_at", ascending=False).head(10)
    else:
        incidents_df = pd.DataFrame([{"message": "Sin incidentes"}])

    runbooks_df = pd.DataFrame(stores["runbooks"]).copy()
    if not runbooks_df.empty:
        runbooks_df["started_at"] = pd.to_datetime(runbooks_df["started_at"], errors="coerce", utc=True)
        runbooks_df = runbooks_df.sort_values("started_at", ascending=False).head(10)
    else:
        runbooks_df = pd.DataFrame([{"message": "Sin runbooks"}])

    print("Alertas recientes")
    display(alerts_df)
    print("Incidentes recientes")
    display(incidents_df)
    print("Runbooks recientes")
    display(runbooks_df)


def show_dashboard():
    if WIDGETS_AVAILABLE:
        max_window = max(50, min(500, len(get_predictions_df())))
        window_slider = widgets.IntSlider(
            value=min(220, max_window),
            min=50,
            max=max_window,
            step=10,
            description="Ventana:"
        )

        incident_options = ["All"] + sorted(
            set(
                str(x)
                for x in get_predictions_df().get("incident_tag", pd.Series(dtype=str)).dropna().unique()
            )
        )

        incident_dropdown = widgets.Dropdown(
            options=incident_options,
            value="All",
            description="Incidente:"
        )

        out = widgets.Output()

        def _render(change=None):
            with out:
                clear_output(wait=True)
                render_advanced_dashboard(
                    window=window_slider.value,
                    incident_filter=incident_dropdown.value
                )

        window_slider.observe(_render, names="value")
        incident_dropdown.observe(_render, names="value")

        display(widgets.HBox([window_slider, incident_dropdown]), out)
        _render()
    else:
        render_advanced_dashboard()

show_dashboard()

Output()